# Power BI — Conexión y Dashboards

Este notebook cumple dos funciones:

1. **Documenta la conexión** de Power BI Desktop al SQL Warehouse de Databricks
   (requerimiento de la guía, sección 3.5 del PDF oficial).
2. **Construye 3 vistas analíticas pre-agregadas** sobre la capa Gold que alimentan
   los tres dashboards exigidos: ejecutivo, operacional y de calidad de datos.

Las vistas viven en el schema `gold` y se llaman `vw_dashboard_*`. Power BI se
conecta a ellas con **Direct Query** (datos siempre frescos) o **Import Mode**
(mejor rendimiento de navegación).

## Parte A — Conexión de Power BI Desktop a Databricks

### Paso 1. Obtener credenciales del SQL Warehouse

En Databricks:

1. Menú izquierdo → **SQL Warehouses**.
2. Si no hay ninguno: **Create SQL Warehouse** → tipo **Serverless** → tamaño **2X-Small** → **Create**.
3. Abrir el warehouse activo → pestaña **Connection details**.
4. Copiar:
   - **Server hostname** (ej. `dbc-xxxxx.cloud.databricks.com`)
   - **HTTP path** (ej. `/sql/1.0/warehouses/abc123def456`)

### Paso 2. Generar Personal Access Token (PAT)

1. Esquina superior derecha → **Settings** (ícono de usuario) → **User Settings**.
2. Pestaña **Developer** → **Access tokens** → **Generate new token**.
3. Comment: `power-bi-wanderbricks`, Lifetime: 90 days → **Generate**.
4. **Copiar el token** (no se vuelve a mostrar).

### Paso 3. Conectar desde Power BI Desktop

1. Abrir **Power BI Desktop** → **Obtener datos** → buscar **Azure Databricks** → **Conectar**.
2. Pegar **Server hostname** y **HTTP Path**.
3. **Data Connectivity mode**: elegir **DirectQuery** (recomendado para datos frescos)
   o **Import** (mejor rendimiento si los datos no cambian mucho).
4. Autenticación: **Personal Access Token** → pegar el PAT del paso 2.
5. En el navegador, expandir catálogo → schema **gold** → seleccionar las vistas
   `vw_dashboard_ejecutivo`, `vw_dashboard_operacional`, `vw_dashboard_calidad`
   y las tablas dimensionales (`gold_dim_*`) → **Cargar**.

## Parte B — Vistas analíticas para los dashboards

Las siguientes celdas crean las vistas SQL pre-agregadas. Hacer **Run all** una
sola vez antes de abrir Power BI.

## Dashboard 1 — Ejecutivo (KPIs, tendencias, comparativos)

Vista que entrega métricas agregadas a alto nivel: ingresos totales, número de
reservas, ticket promedio, noches promedio, distribución geográfica. Una fila
por combinación país–año–mes.

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_dashboard_ejecutivo AS
SELECT
  t.year,
  t.quarter,
  t.month,
  t.month_name,
  d.country                                   AS pais_destino,
  d.destination_name                          AS destino,
  COUNT(*)                                    AS total_reservas,
  ROUND(SUM(f.total_amount), 2)               AS ingresos_totales,
  ROUND(AVG(f.total_amount), 2)               AS ticket_promedio,
  ROUND(AVG(f.total_nights), 2)               AS noches_promedio,
  SUM(f.total_nights)                         AS noches_totales,
  ROUND(SUM(f.payment_amount), 2)             AS pagos_recibidos,
  COUNT(DISTINCT f.user_id)                   AS usuarios_unicos,
  COUNT(DISTINCT f.property_id)               AS propiedades_reservadas
FROM gold.gold_fact_reservas f
JOIN gold.gold_dim_time t          ON f.tiempo_id      = t.tiempo_id
JOIN gold.gold_dim_destinations d  ON f.destination_id = d.destination_key
GROUP BY t.year, t.quarter, t.month, t.month_name, d.country, d.destination_name;

In [ ]:
%sql
-- Validación visual de la vista ejecutiva
SELECT * FROM gold.vw_dashboard_ejecutivo
ORDER BY ingresos_totales DESC
LIMIT 10;

### KPIs sugeridos para el dashboard ejecutivo

En Power BI, configurar las siguientes visualizaciones a partir de
`vw_dashboard_ejecutivo`:

| Visualización | Campos sugeridos |
|---|---|
| Tarjeta — Ingresos totales | `SUM(ingresos_totales)` |
| Tarjeta — Reservas totales | `SUM(total_reservas)` |
| Tarjeta — Ticket promedio | `AVG(ticket_promedio)` |
| Tarjeta — Usuarios únicos | `SUM(usuarios_unicos)` |
| Línea — Ingresos por mes | eje X: `year + month`, eje Y: `ingresos_totales` |
| Mapa — Reservas por país | ubicación: `pais_destino`, tamaño: `total_reservas` |
| Barras — Top 10 destinos | eje: `destino`, valor: `ingresos_totales` |
| Filtros (slicers) | `year`, `quarter`, `pais_destino` |

## Dashboard 2 — Operacional (drill-down a nivel reserva)

Vista detallada con filtros funcionales por fecha, tipo de propiedad y estado.
Una fila por reserva enriquecida con atributos descriptivos de usuario,
propiedad y destino.

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_dashboard_operacional AS
SELECT
  f.booking_id,
  f.check_in,
  f.check_out,
  f.total_nights,
  f.guests_count,
  f.total_amount,
  f.payment_amount,
  f.booking_status,
  u.name                AS usuario,
  u.country             AS pais_usuario,
  u.user_type,
  u.is_business,
  p.title               AS propiedad,
  p.property_type,
  p.max_guests,
  p.bedrooms,
  p.base_price,
  d.destination_name    AS destino,
  d.country             AS pais_destino,
  d.state_or_province   AS region,
  t.year,
  t.quarter,
  t.month_name,
  t.day_name,
  t.is_weekend
FROM gold.gold_fact_reservas f
LEFT JOIN gold.gold_dim_users        u ON f.user_id        = u.user_key
LEFT JOIN gold.gold_dim_properties   p ON f.property_id    = p.property_key
LEFT JOIN gold.gold_dim_destinations d ON f.destination_id = d.destination_key
LEFT JOIN gold.gold_dim_time         t ON f.tiempo_id      = t.tiempo_id;

In [ ]:
%sql
-- Validación visual de la vista operacional
SELECT * FROM gold.vw_dashboard_operacional
ORDER BY check_in DESC
LIMIT 10;

### Visualizaciones sugeridas para el dashboard operacional

| Visualización | Campos sugeridos |
|---|---|
| Tabla detallada | todas las columnas, con búsqueda y ordenamiento |
| Matriz drill-down | filas: `pais_destino > destino > propiedad`, valor: `SUM(total_amount)` |
| Barras apiladas | eje: `property_type`, valor: `total_reservas`, leyenda: `booking_status` |
| Donut — Estado de reservas | `booking_status` |
| Donut — Tipo de usuario | `user_type` (individual vs business) |
| Slicers | `year`, `month_name`, `pais_destino`, `property_type`, `booking_status`, `is_weekend` |

## Dashboard 3 — Calidad de Datos (completitud, frescura, anomalías)

Vista que reporta métricas de salud del pipeline: completitud de Silver vs
Bronze, frescura por capa, ratio de reservas con pagos asociados, anomalías
de montos.

In [ ]:
%sql
CREATE OR REPLACE VIEW gold.vw_dashboard_calidad AS
SELECT
  'bookings'    AS entidad,
  (SELECT COUNT(*) FROM bronze.bronze_bookings)     AS registros_bronze,
  (SELECT COUNT(*) FROM silver.silver_bookings)     AS registros_silver,
  ROUND(
    (SELECT COUNT(*) FROM silver.silver_bookings) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM bronze.bronze_bookings), 0),
  2)                                                AS porcentaje_retenido
UNION ALL
SELECT 'users',     (SELECT COUNT(*) FROM bronze.bronze_users),
                    (SELECT COUNT(*) FROM silver.silver_users),
                    ROUND((SELECT COUNT(*) FROM silver.silver_users) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_users), 0), 2)
UNION ALL
SELECT 'properties',(SELECT COUNT(*) FROM bronze.bronze_properties),
                    (SELECT COUNT(*) FROM silver.silver_properties),
                    ROUND((SELECT COUNT(*) FROM silver.silver_properties) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_properties), 0), 2)
UNION ALL
SELECT 'payments', (SELECT COUNT(*) FROM bronze.bronze_payments),
                    (SELECT COUNT(*) FROM silver.silver_payments),
                    ROUND((SELECT COUNT(*) FROM silver.silver_payments) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_payments), 0), 2)
UNION ALL
SELECT 'reviews',  (SELECT COUNT(*) FROM bronze.bronze_reviews),
                    (SELECT COUNT(*) FROM silver.silver_reviews),
                    ROUND((SELECT COUNT(*) FROM silver.silver_reviews) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_reviews), 0), 2)
UNION ALL
SELECT 'destinations',(SELECT COUNT(*) FROM bronze.bronze_destinations),
                    (SELECT COUNT(*) FROM silver.silver_destinations),
                    ROUND((SELECT COUNT(*) FROM silver.silver_destinations) * 100.0 / NULLIF((SELECT COUNT(*) FROM bronze.bronze_destinations), 0), 2);

In [ ]:
%sql
SELECT * FROM gold.vw_dashboard_calidad
ORDER BY entidad;

In [ ]:
%sql
-- Vista adicional: frescura y anomalías financieras
CREATE OR REPLACE VIEW gold.vw_dashboard_calidad_detalle AS
SELECT
  MIN(check_in)                                  AS fecha_minima_reserva,
  MAX(check_in)                                  AS fecha_maxima_reserva,
  DATEDIFF(MAX(check_in), MIN(check_in))         AS rango_dias,
  COUNT(*)                                       AS total_reservas,
  COUNT(CASE WHEN payment_amount = 0 THEN 1 END) AS reservas_sin_pago,
  ROUND(COUNT(CASE WHEN payment_amount = 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_sin_pago,
  COUNT(CASE WHEN total_amount > 5000 THEN 1 END) AS reservas_alto_monto,
  COUNT(CASE WHEN total_nights > 30 THEN 1 END)   AS reservas_estancia_larga
FROM gold.gold_fact_reservas;

In [ ]:
%sql
SELECT * FROM gold.vw_dashboard_calidad_detalle;

### Visualizaciones sugeridas para el dashboard de calidad

| Visualización | Campos sugeridos | Vista |
|---|---|---|
| Tabla — Retención por entidad | `entidad`, `registros_bronze`, `registros_silver`, `porcentaje_retenido` | `vw_dashboard_calidad` |
| Barras horizontales — % retenido | eje: `entidad`, valor: `porcentaje_retenido` | `vw_dashboard_calidad` |
| Tarjeta — Reservas totales | `total_reservas` | `vw_dashboard_calidad_detalle` |
| Tarjeta — Reservas sin pago | `reservas_sin_pago` + `pct_sin_pago` | `vw_dashboard_calidad_detalle` |
| Tarjeta — Rango temporal de datos | `fecha_minima_reserva` → `fecha_maxima_reserva` | `vw_dashboard_calidad_detalle` |
| Tarjeta — Reservas anómalas | `reservas_alto_monto`, `reservas_estancia_larga` | `vw_dashboard_calidad_detalle` |

## Conclusión

Quedaron creadas 4 vistas analíticas en el schema `gold`:

- `gold.vw_dashboard_ejecutivo`
- `gold.vw_dashboard_operacional`
- `gold.vw_dashboard_calidad`
- `gold.vw_dashboard_calidad_detalle`

Power BI Desktop se conecta al SQL Warehouse de Databricks usando el conector
nativo (Azure Databricks) con DirectQuery o Import Mode, y consume estas vistas
para construir los 3 dashboards exigidos por la guía.

Decisiones de diseño defendibles ante el docente:

- **¿Por qué vistas y no medidas DAX?** Mover la lógica al lado del warehouse
  reduce la carga de Power BI, aprovecha el motor distribuido de Spark SQL y
  garantiza que cualquier herramienta (Power BI, Tableau, Excel) vea las mismas
  cifras.
- **¿Por qué Direct Query vs Import?** Direct Query mantiene los dashboards
  sincronizados con el streaming en tiempo real (sin refrescos manuales).
  Import es preferible si los datos no cambian frecuentemente y se prioriza
  velocidad de interacción.
- **¿Por qué un dashboard de calidad propio?** Hace observable la salud del
  pipeline y permite detectar regresiones (cambios de % retenido entre cargas,
  picos de reservas sin pago, etc.) sin tener que bajar al código.